# Working with Dates and Times

In [ ]:
import pandas as pd

## Review of Python's datetime Module
- A module is a file of Python code.
- The `datetime` module is part of Python's standard library, a collection of built-in features.
- The common alias for the `datetime` module is `dt`.

In [ ]:
import datetime as dt

- The `datetime` module exports `date`, `time`, and `datetime` classes for representing dates and datetimes.
- The `date` constructor accepts arguments for year, month, and day.
- The `time` constructor accepts arguments for hours, minutes, and seconds.
- The `datetime` constructor accepts arguments for year, month, day, hour, minute, and second.
- The constructors default to 0 for any missing values.

## The dt Attribute
- The `dt` attribute on a `Series` exposes a `DatetimeProperties` object with attributes/methods for working with datetimes.
- The `DatetimeProperties` object has attributes like `day`, `month`, and `year` to reveal information about each date in the `Series`.

In [ ]:
ibm = pd.read_csv("ibm.csv", parse_dates=["Date"])
ibm.dtypes

In [ ]:
ibm["Date"].dt.day

- The `day_name` method returns the written day of the week.

In [ ]:
ibm["Date"].dt.day_name()

- Attributes like `is_month_end` and `is_quarter_start` return Boolean `Series`.

In [ ]:
ibm[ibm["Date"].dt.is_month_end]

## The Timestamp Type
- Pandas has its own `Timestamp` type for modelling a date or datetime.

In [ ]:
pd.Timestamp(2026, 12, 15)

- The `Timestamp` constructor also accepts `dt.date` and `dt.datetime` objects.
- The `Timestamp` cannot model a time without a day and thus does not accept `dt.time` objects.

- When we convert a column to `datetime64`, each row actually stores a `Timestamp`.

## Extracting Datetime Rows from a DataFrame
- When generating a Boolean `Series` against a `datetime64` column, you can use either strings, `datetime` objects, or `Timestamp` objects.

In [ ]:
stocks = pd.read_csv("ibm.csv", parse_dates=["Date"]).sort_values("Date")
stocks.head()

In [ ]:
stocks[stocks["Date"] == pd.Timestamp(1995, 4, 3)]
stocks[stocks["Date"] == dt.datetime(1995, 4, 3)]
stocks[stocks["Date"] == pd.Timestamp(dt.datetime(1995, 4, 3))]

- We can set a column of `datetime` values as the index.
- The `DataFrame` sill store a `DatetimeIndex` object.

In [ ]:
stocks = pd.read_csv("ibm.csv", parse_dates=["Date"], index_col="Date").sort_index()
stocks.head()

- Use the `iloc` accessor for index position-based extraction.

In [ ]:
stocks.iloc[2]

- Use the `loc` accessor for index label-based extraction.
- The accessor accepts strings, `datetime` objects, or `Timestamp` objects.
- Use list slicing to extract a sequence of dates.
- The `truncate` method is another alternative for extracting a range of dates..

In [ ]:
stocks.truncate("2022-03-04", "2023-01-01")

- The second argument to `loc` represents the column(s) to target.

- Pass a shorter string like `"2025"` or `"2025-03"` to `loc` to select all rows within that year or month.

In [ ]:
stocks.loc["2025"]
stocks.loc["2025-01"]

## The DateOffset Object
- A `DateOffset` object adds time to a `Timestamp` to arrive at a new `Timestamp`.
- The `DateOffset` constructor accepts `days`, `weeks`, `months`, `years` parameters, and more.
- We can pass a `DateOffset` object to the `freq` parameter of the `pd.date_range` function.

In [ ]:
stocks = pd.read_csv("ibm.csv", parse_dates=["Date"], index_col="Date").sort_index()
stocks.head()

In [ ]:
pd.Timestamp(2026, 2, 3) + pd.DateOffset(weeks=5, days=3)

In [ ]:
stocks.index + pd.DateOffset(years=1, months=3, weeks=2, days=1)

## The date_range Function
- The `pd.date_range` function generates a `MultiIndex` of values from a start point to an endpoint.
- We can customize the frequency/gap between each `Timestamp` with the `freq` parameter.
- We can also ask it to generate a certain number of values with the `periods` parameter and either a start or end.

In [ ]:
pd.date_range(start="2004-07-15", end="15-07-15", freq="4h")

In [ ]:
pd.date_range(end="2004-07-15", freq="7D", periods=200)

- Let's extract all of the rows that fell on a given date each year.

In [ ]:
birthdays = pd.date_range(
    start="2004-07-15", end="2026-07-15", freq=pd.DateOffset(years=1)
)

In [ ]:
stocks[stocks.index.isin(birthdays)]

## Specialized Date Offsets
- Pandas nests more specialized date offsets in `pd.tseries.offsets`.
- The offsets represent custom spans of time (month end, quarter end, year begin, etc).
- Use `+` to move time forward into the future.
- Use `-` to move time backward into the past.

In [ ]:
stocks.index - pd.tseries.offsets.MonthEnd()

In [ ]:
stocks.index + pd.tseries.offsets.QuarterEnd()

## Timedeltas
- A `Timedelta` is a pandas object that represents a duration (an amount of time).
- Subtracting two `Timestamp` objects will yield a `Timedelta` object (this applies to subtracting a `Series` from another `Series`).
- The `Timedelta` constructor accepts parameters for time as well as string descriptions.

In [ ]:
pd.Timestamp("2026-07-15 12:30:40") - pd.Timestamp("2026-06-22 19:40:24")

In [ ]:
pd.Timedelta(days=3, hours=2, minutes=5)

In [ ]:
ecommerce = pd.read_csv(
    "ecommerce.csv",
    index_col="ID",
    parse_dates=["order_date", "delivery_date"],
    date_format="%m/%d/%y",
)
ecommerce.head()

In [ ]:
ecommerce["delivery_time"] = ecommerce["delivery_date"] - ecommerce["order_date"]
ecommerce.head()